In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)

df = pd.read_csv("hotel_bookings_clean.csv")

print("Shape:", df.shape)

df.head()

Shape: (39070, 37)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_guests,total_nights,family,arrival_month_num,season
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0.0,0.0,0.0,C,C,3.0,No Deposit,0.0,NaN,0.0,Transient,0.0,0.0,0.0,Check-Out,2015-07-01,2.0,0.0,0.0,7.0,Summer
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0.0,0.0,0.0,C,C,4.0,No Deposit,0.0,NaN,0.0,Transient,0.0,0.0,0.0,Check-Out,2015-07-01,2.0,0.0,0.0,7.0,Summer
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0.0,0.0,0.0,A,C,0.0,No Deposit,0.0,NaN,0.0,Transient,75.0,0.0,0.0,Check-Out,2015-07-02,1.0,1.0,0.0,7.0,Summer
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0.0,0.0,0.0,A,A,0.0,No Deposit,304.0,NaN,0.0,Transient,75.0,0.0,0.0,Check-Out,2015-07-02,1.0,1.0,0.0,7.0,Summer
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0.0,0.0,0.0,A,A,0.0,No Deposit,240.0,NaN,0.0,Transient,98.0,0.0,1.0,Check-Out,2015-07-03,2.0,2.0,0.0,7.0,Summer


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39070 entries, 0 to 39069
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           39070 non-null  object 
 1   is_canceled                     39070 non-null  int64  
 2   lead_time                       39070 non-null  int64  
 3   arrival_date_year               39070 non-null  int64  
 4   arrival_date_month              39070 non-null  object 
 5   arrival_date_week_number        39070 non-null  int64  
 6   arrival_date_day_of_month       39070 non-null  int64  
 7   stays_in_weekend_nights         39070 non-null  int64  
 8   stays_in_week_nights            39070 non-null  int64  
 9   adults                          39070 non-null  int64  
 10  children                        39070 non-null  float64
 11  babies                          39070 non-null  int64  
 12  meal                            

In [7]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values: 36472
Duplicate rows: 0


In [8]:
df['total_guests'] = (
    df['adults'] +
    df['children'] +
    df['babies']
)

In [9]:
df['total_nights'] = (
    df['stays_in_weekend_nights'] +
    df['stays_in_week_nights']
)

In [10]:
df['is_family'] = np.where(
    (df['children'] > 0) | (df['babies'] > 0),
    1,
    0
)

In [11]:
df['has_weekend_stay'] = np.where(
    df['stays_in_weekend_nights'] > 0,
    1,
    0
)

In [12]:
df['estimated_revenue'] = (
    df['adr'] *
    df['total_nights']
)

In [13]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'


df['season'] = df['arrival_month_num'].apply(get_season)

In [14]:
df['season'].value_counts()

,count
season,
Summer,12910
Spring,9400
Autumn,8969
Winter,7791


In [15]:
def booking_size(guests):

    if guests == 1:
        return 'Single'

    elif guests == 2:
        return 'Couple'

    elif guests <= 4:
        return 'Small Group'

    else:
        return 'Large Group'


df['booking_size'] = df['total_guests'].apply(booking_size)

In [16]:
df['booking_size'].value_counts()

,count
booking_size,
Couple,26726
Single,7007
Small Group,5273
Large Group,64


In [17]:
def stay_category(nights):

    if nights <= 2:
        return 'Short Stay'

    elif nights <= 5:
        return 'Medium Stay'

    else:
        return 'Long Stay'


df['stay_category'] = df['total_nights'].apply(stay_category)

In [18]:
def lead_time_category(days):

    if days <= 7:
        return 'Last Minute'

    elif days <= 30:
        return 'Short Lead Time'

    elif days <= 90:
        return 'Medium Lead Time'

    else:
        return 'Long Lead Time'


df['lead_time_category'] = df['lead_time'].apply(
    lead_time_category
)

In [19]:
df['lead_time_category'].value_counts()

,count
lead_time_category,
Long Lead Time,13028
Last Minute,9540
Medium Lead Time,9502
Short Lead Time,7000


In [20]:
leakage_columns = [
    'reservation_status',
    'reservation_status_date'
]

df_ml = df.drop(
    columns=leakage_columns,
    errors='ignore'
)

In [21]:
df_ml = df_ml.drop(
    columns=['agent'],
    errors='ignore'
)

In [22]:
df.drop(columns=['company'])

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_guests,total_nights,family,arrival_month_num,season,is_family,has_weekend_stay,estimated_revenue,booking_size,stay_category,lead_time_category
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0.0,0.0,0.0,C,C,3.0,No Deposit,0.0,0.0,Transient,0.00,0.0,0.0,Check-Out,2015-07-01,2.0,0,0.0,7.0,Summer,0,0,0.00,Couple,Short Stay,Long Lead Time
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0.0,0.0,0.0,C,C,4.0,No Deposit,0.0,0.0,Transient,0.00,0.0,0.0,Check-Out,2015-07-01,2.0,0,0.0,7.0,Summer,0,0,0.00,Couple,Short Stay,Long Lead Time
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0.0,0.0,0.0,A,C,0.0,No Deposit,0.0,0.0,Transient,75.00,0.0,0.0,Check-Out,2015-07-02,1.0,1,0.0,7.0,Summer,0,0,75.00,Single,Short Stay,Last Minute
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0.0,0.0,0.0,A,A,0.0,No Deposit,304.0,0.0,Transient,75.00,0.0,0.0,Check-Out,2015-07-02,1.0,1,0.0,7.0,Summer,0,0,75.00,Single,Short Stay,Short Lead Time
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0.0,0.0,0.0,A,A,0.0,No Deposit,240.0,0.0,Transient,98.00,0.0,1.0,Check-Out,2015-07-03,2.0,2,0.0,7.0,Summer,0,0,196.00,Couple,Short Stay,Short Lead Time
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39065,City Hotel,0,94,2016,April,15,5,0,3,2,0.0,0,BB,BEL,Direct,Direct,0.0,0.0,0.0,D,D,0.0,No Deposit,14.0,0.0,Transient,90.90,0.0,0.0,Check-Out,2016-04-08,2.0,3,0.0,4.0,Spring,0,0,272.70,Couple,Medium Stay,Long Lead Time
39066,City Hotel,1,42,2016,April,15,5,0,3,1,0.0,0,BB,SAU,Online TA,TA/TO,0.0,0.0,0.0,D,D,0.0,No Deposit,9.0,0.0,Transient,131.40,0.0,0.0,Canceled,2016-03-28,1.0,3,0.0,4.0,Spring,0,0,394.20,Single,Medium Stay,Medium Lead Time
39067,City Hotel,1,78,2016,April,15,5,0,4,2,0.0,0,BB,HUN,Online TA,TA/TO,0.0,0.0,0.0,D,D,0.0,No Deposit,9.0,0.0,Transient,97.33,0.0,0.0,Canceled,2016-03-18,2.0,4,0.0,4.0,Spring,0,0,389.32,Couple,Medium Stay,Medium Lead Time
39068,City Hotel,1,120,2016,April,15,5,0,4,2,0.0,0,BB,PRT,Online TA,TA/TO,0.0,0.0,0.0,E,E,0.0,No Deposit,11.0,0.0,Transient,94.86,0.0,0.0,Canceled,2016-03-22,2.0,4,0.0,4.0,Spring,0,0,379.44,Couple,Medium Stay,Long Lead Time


In [23]:
X = df_ml.drop(
    columns=['is_canceled']
)

y = df_ml['is_canceled']

In [24]:
print("Features:", X.shape)
print("Target:", y.shape)

Features: (39070, 39)
Target: (39070,)


In [25]:
categorical_features = X.select_dtypes(
    include=['object']
).columns.tolist()

categorical_features

['hotel',
 'arrival_date_month',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'reserved_room_type',
 'assigned_room_type',
 'deposit_type',
 'customer_type',
 'season',
 'booking_size',
 'stay_category',
 'lead_time_category']

In [26]:
numerical_features = X.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

numerical_features

['lead_time',
 'arrival_date_year',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'booking_changes',
 'company',
 'days_in_waiting_list',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'total_guests',
 'total_nights',
 'family',
 'arrival_month_num',
 'is_family',
 'has_weekend_stay',
 'estimated_revenue']

In [27]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [28]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(
                handle_unknown='ignore'
            ),
            categorical_features
        )
    ],
    remainder='passthrough'
)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [30]:
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target:")
print(y_train.value_counts(normalize=True))

print("\nTesting target:")
print(y_test.value_counts(normalize=True))

Training data: (31256, 39)
Testing data: (7814, 39)

Training target:
is_canceled
0    0.746161
1    0.253839
Name: proportion, dtype: float64

Testing target:
is_canceled
0    0.746225
1    0.253775
Name: proportion, dtype: float64


In [31]:
df.to_csv(
    "hotel_bookings_engineered.csv",
    index=False
)

In [32]:
df_ml.to_csv(
    "hotel_bookings_ml.csv",
    index=False
)

In [33]:
print("=" * 50)
print("FINAL DATASET CHECK")
print("=" * 50)

print("Rows:", df_ml.shape[0])
print("Columns:", df_ml.shape[1])
print("Missing values:", df_ml.isnull().sum().sum())
print("Duplicates:", df_ml.duplicated().sum())
print("Target distribution:")
print(y.value_counts())

FINAL DATASET CHECK
Rows: 39070
Columns: 40
Missing values: 36467
Duplicates: 118
Target distribution:
is_canceled
0    29153
1     9917
Name: count, dtype: int64
